<div align="right"><a href="S19-Design-of-a-Hilbert-transformer-Remez-algorithm.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L19-Entwurf-eines-Hilbert-Transformators-%28Remez-Verfahren%29.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L17 Remez-Entwurf (Tschebyscheff-Approximation)](./L17-Remez-Entwurf-%28Tschebyscheff-Approximation%29.ipynb) | [Aufgabe 19](./A19-Entwurf-eines-Hilbert-Transformators-%28Remez-Verfahren%29.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L20 Eigenschaften der Diskreten Fouriertransformation (DFT)](./L20-Diskrete-Fouriertransformation-%28DFT%29.ipynb)

</div>

---


### Aufgabe: Entwurf eines Hilbert-Transformators

In dieser Aufgabe soll ein Hilbert-Transformator mit idealer Frequenzantwort

$$
H(e^{j\Omega}) = j\,\mathrm{sgn}(\Omega)
$$

mithilfe des Remez-Algorithmus (`remez()` in MATLAB) approximiert werden.

---

#### a) 

Entwerfen Sie zwei Hilbert-Transformatoren der Ordnung 30 und 31 unter Verwendung des zusätzlichen Arguments `'h'`:

```matlab
b = remez(m, F, G, 'h');
````

Plotten Sie die Amplituden- und Impulsantwort sowie die Pol-Nullstellen-Ebene. Welche Filtertypen wurden erstellt?

---

#### b)

Erstellen Sie ein Sinussignal mit einer Dauer von zwei Sekunden, einer Signalfrequenz von $50\text{Hz}$ und einer Abtastfrequenz von $1000\text{Hz}$.

Filtern Sie dieses Signal mit dem Hilbert-Transformator der Ordnung $m=30$ und plotten Sie Eingangs- und Ausgangssignal.

*Hinweis:* Die Gruppenverzögerung muss berücksichtigt werden, da das Ausgangssignal verzögert ist.

---

#### c)

Für die Hilbert-Transformation (wichtig in der Kommunikationstechnik) bietet MATLAB eine eigene Funktion:

```matlab
y = hilbert(x);
```

Der Rückgabewert `y` ist das analytische Signal zu `x`, der Realteil von `y` entspricht `x` und der Imaginärteil von `y` ist die Hilbert-Transformation von `x`.

Berechnen Sie die Hilbert-Transformation des Signals aus Aufgabe **b** und vergleichen Sie die Ergebnisse.

In [ ]:
# BEGIN SOLUTION
% #####  Teilaufgabe a:  Hilbert-Transformator entwerfen  #####
b1 = remez(30,[0.05 0.95],[1 1],'h');
b2 = remez(31,[0.05 1],[1 1],'h');

[H1 W1] = freqz(b1,1,512);
[H2 W2] = freqz(b2,1,512);

% # grafische Ausgabe # %
figure; 
plot(W1/pi,abs(H1)); 
grid; 
xlabel('Omega/pi');
ylabel('|H_H(exp(j*Omega))|'); 
title('Filterordnung: 30 Typ II');

figure; 
plot(W2/pi,abs(H2)); 
grid; 
xlabel('Omega/pi');
ylabel('|H_H(exp(j*Omega))|'); 
title('Filterordnung: 31 Typ IV');

figure; 
stem([0:30], b1); 
axis([0 30 -0.8 0.8]);     
grid; 
xlabel('k');
ylabel('Impulsantwort');       
title('Filterordnung: 30 Typ II');

figure; 
stem([0:31], b2); 
axis([0 31 -0.8 0.8]);     
grid; 
xlabel('k');
ylabel('Impulsantwort');       
title('Filterordnung: 31 Typ IV');

% #####  Teilaufgabe b:  Ausgangssignal filtern  #####
b  = remez(30,[0.05 0.95],[1 1],'h');
Fs = 1000;
t  = (0:1/Fs:2)';
x  = sin(2*pi*50*t);
xh = filter(b,1,x);

figure;
zplane(b); 
xlabel('Realteil'); 
ylabel('Imaginaerteil');
title('Pol-Nullstellen-Diagramm');

xd = [zeros(15,1); x(1:length(x)-15)];  % delay 15 samples

figure; 
plot(t(1:50),xd(1:50),t(1:50),xh(1:50),'--'); 
grid; 
xlabel('Zeit (sek)');
ylabel('Amplitude'); 
title(' - Eingangssig.   -- Ausgangssig.');

% #####  Teilaufgabe c:  Hilbert-Tranformierte berechnen  #####
Fs = 1000;
t  = (0:1/Fs:2)';
x  = sin(2*pi*50*t);
y = hilbert(x);

figure; 
plot(t(1:50), real(y(1:50))); 
grid; 
hold on;
plot(t(1:50), imag(y(1:50)),'--'); 
xlabel('Zeit (sek)'); 
ylabel('Amplitude');
title('-- Hilbert-Transform.   - Original');
% #####  EOF  #####

# END SOLUTION

# Lösung: Entwurf eines Hilbert-Transformators

### a) Filterentwurf und Impuls-/Amplitudenantwort

Wenn wir den optionalen Parameter[^1] berücksichtigen, erhalten wir einen Filter vom Typ II für die Filterordnung $m=30$ und einen Filter vom Typ IV für die Filterordnung $m=31$. Die folgenden Abbildungen zeigen die Amplituden- und Impulsantworten der Filterentwürfe.

<img src="./pngs/ldsv5a05_a1.png" width="250">
<img src="./pngs/ldsv5a05_a3.png" width="250">
<img src="./pngs/ldsv5a05_a2.png" width="250">
<img src="./pngs/ldsv5a05_a4.png" width="250">

Es zeigt sich außerdem, dass die Annäherungsqualität bei ungerader Filterordnung höher ist, da die Nullstelle bei $\Omega = \pi$ fehlt.

Um einen Hilbert-Transformator zur Erzeugung analytischer Signale zu verwenden, müssen das Ausgangssignal (Imaginärteil) und das Eingangssignal (Realteil) gleich verzögert und kombiniert werden. Das bedeutet, dass eine gerade Gruppendelay existieren sollte. Bei einem nichtrekursiven, linearen System trägt jede Nullstelle $-0.5$ zur Gruppendelay bei. Daher müsste im Fall eines Hilbert-Transformators eine gerade Filterordnung existieren. Dies widerspricht jedoch der Möglichkeit, eine hohe Annäherungsqualität zu erzielen.

Zur Verdeutlichung der Nullstellenposition zeigt die folgende Abbildung die Nullstellenverteilung des Filters der Ordnung $m=30$.

<img src="./pngs/ldsv5a05_a5.png" width="250">

Es gibt $15$ Nullstellen, die entweder auf dem Einheitskreis liegen oder als Paar spiegelbildlich auf dem Einheitskreis angeordnet sind, da das System linear und nichtrekursiv ist.

[^1]: Wenn dieser Parameter nicht festgelegt ist, wird der Filter vom Typ I/III erzeugt. Das Design ist daher nicht impulse-invariant, siehe Aufgabe 5.4.a.

---

### b) Gruppendelay und Signale

Wenn man die Gruppendelay bei der Signalbeschreibung berücksichtigt, ergibt sich eine Verzögerung von $15$ bei einer Filterordnung von $m=30$, entsprechend Aufgabe a).  

Die folgende Abbildung überprüft die Beziehung  

$$
-\cos(x) = \mathcal{H}\{\sin(x)\}
$$  

für eine Sinusreihe mit der Hilbert-Transformation $\mathcal{H}\{\cdot\}$.

<img src="./pngs/ldsv5a05_b1.png" width="250">

---

### c) Analytisches Signal mit `hilbert()`

Verwendet man die Funktion `hilbert()`, um ein analytisches Signal zu berechnen, erhält man für die Sinusreihe aus Aufgabe a) die folgende Darstellung:

<img src="./pngs/ldsv5a05_c1.png" width="250">

Ein Vergleich mit den Ergebnissen aus Aufgabe b) zeigt, dass das Prinzip identisch ist, lediglich die Gruppendelay in den Abbildungen unterschiedlich ist.